# Experiment 13 — A pocket niche that partly resolves with inflammation

**Question.** In Exps 09–12 the pocket supports the subgingival community through a fixed term $\gamma D$: once attachment is lost, the niche stays the same size whether the site is inflamed or not. Clinically, non-surgical therapy reduces probing depth by 1–2 mm, largely because inflammation and oedema resolve, while lost attachment does not return. Here part of the niche resolves with inflammation:

$$\frac{dM}{dt}=r\left[-M+G(H)+\gamma D\,\phi(H)\right],\qquad \phi(H)=c+(1-c)H .$$

$c$ is the fraction of the niche that persists when inflammation has resolved (the residual defect). $c=1$ recovers Exps 09–12 exactly; $c=0.5$ and $c=0.25$ let half or three quarters of the niche resolve. In an inflamed site ($H\approx0.95$) the niche is almost unchanged, so the comparison isolates what happens when inflammation falls.

**What it tests**
* whether the three stages of attachment loss ($D_{irr}$, $D_{spont}$) and their correspondence to the steps of therapy survive (tension #9 of the plausibility audit);
* how far hygiene, one debridement and repeated debridement reach, and what maintenance tolerates after therapy;
* whether, with random plaque peaks and no therapy, sites can now remit spontaneously, and whether activity becomes episodic without treatment (the open part of tension #14). Peaks are the same random sequences as Exp 12.

**Sections**
* **A** stage boundaries $D_{irr}(c)$, $D_{spont}(c)$ and equilibria
* **B** treatment ladder at fixed attachment loss
* **C** maintenance after therapy
* **D** random plaque peaks: no therapy, and treat on detection
* **E** checks (including exact reproduction of Exps 09 and 12 at $c=1$), summary and export

Runtime in Colab: about 5–10 minutes.

In [ ]:
# =====================================================================
# EXPERIMENT 13 — folder setup
# =====================================================================
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/Biosemiotic_Perio")
except ImportError:                      # local execution
    BASE_DIR = Path.cwd() / "Biosemiotic_Perio"
EXP_DIR = BASE_DIR / "Experiment_13_Resolving_Niche"
EXP_DIR.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("EXP_QUICK") == "1"     # set True for a fast smoke test with reduced grids
print(f"Experiment folder:\n{EXP_DIR}\nExists: {EXP_DIR.exists()}   QUICK={QUICK}")

In [ ]:
import sys, json, time, platform, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import brentq

versions_text = f"""
Experiment: 13 Resolving niche
Created: {datetime.now(timezone.utc).isoformat()}

Python: {sys.version}
Platform: {platform.platform()}
NumPy: {np.__version__}
Pandas: {pd.__version__}
SciPy: {scipy.__version__}
Matplotlib: {matplotlib.__version__}
"""
with open(EXP_DIR / "software_versions.txt", "w") as f:
    f.write(versions_text)
print(versions_text)

# ---- figure style (same tokens as Experiment 07) ------------------------
INK, INK2, GRIDC, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
MEM_COL = {0.3: "#86b6ef", 0.1: "#5598e7", 0.03: "#2a78d6", 0.01: "#1c5cab", 0.003: "#0d366b"}
INSTANT_COL = "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.6,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "legend.frameon": False,
})
def save_fig(fig, name):
    fig.savefig(EXP_DIR / f"{name}.jpg", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =====================================================================
# CORE MODEL — state (H, M, A, D); niche gamma*D*phi(H), phi = c + (1-c)H
# =====================================================================
#   dH/dt = -H + F(M + u(t); theta0*exp(-A))
#   dM/dt =  r*(-M + G(H) + gamma*D*phi(H))       niche: residual part c, inflammatory part (1-c)H
#   dA/dt =  eps*(a*H - A)                        response-state memory ('adaptive')
#   dD/dt =  delta*max(H - hD, 0)*(1 - D)         irreversible, saturating tissue loss
# mode 'adaptive' | 'instant' (A = a*H). c = 1 is the model of Exps 09-12.
COMPARATOR = dict(n=4.0, q=4.0, K=0.9, th0=0.8, r=1.0, a=1.0)   # Exp 06B backbone
DT = 0.1
U_UP, U_DOWN = 0.4682392215, 0.0630204768                       # static folds (D = 0)


def hill(x, h, n):
    x = np.maximum(x, 0.0)
    xn = x ** n
    return xn / (xn + h ** n)


def phi(H, c):
    return c + (1.0 - c) * H


def rhs(H, M, A, D, u, eps, p, mode, gamma, delta, hD, c):
    A_eff = p['a'] * H if mode == 'instant' else A
    dH = -H + hill(M + u, p['th0'] * np.exp(-A_eff), p['n'])
    dM = p['r'] * (-M + hill(H, p['K'], p['q']) + gamma * D * phi(H, c))
    dA = eps * (p['a'] * H - A) if mode == 'adaptive' else 0.0 * A
    dD = delta * np.maximum(H - hD, 0.0) * (1.0 - D)
    return dH, dM, dA, dD


def run(y0, u_fn, n_steps, eps=0.0, mode='adaptive', p=COMPARATOR, gamma=0.0, delta=0.0, hD=0.8, c=1.0,
        events=None, recorder=None, dt=DT):
    """Vectorised fixed-step RK4. u_fn(i, c): forcing at step i, stage offset in {0, .5, 1};
    events(i, H, M, A, D) -> state at the start of step i; recorder(i, H, M, A, D) after each step."""
    H, M, A, D = [np.array(v, float, copy=True) for v in np.broadcast_arrays(*y0)]
    eps, gamma, delta, c = [np.broadcast_to(np.asarray(v, float), H.shape) for v in (eps, gamma, delta, c)]
    if isinstance(p.get('r'), np.ndarray):
        p = dict(p, r=np.broadcast_to(p['r'], H.shape))
    f = lambda h, m, a, d, u: rhs(h, m, a, d, u, eps, p, mode, gamma, delta, hD, c)
    for i in range(n_steps):
        if events is not None:
            H, M, A, D = events(i, H, M, A, D)
        u0, uh, u1 = u_fn(i, 0.0), u_fn(i, 0.5), u_fn(i, 1.0)
        k1 = f(H, M, A, D, u0)
        k2 = f(H + .5*dt*k1[0], M + .5*dt*k1[1], A + .5*dt*k1[2], D + .5*dt*k1[3], uh)
        k3 = f(H + .5*dt*k2[0], M + .5*dt*k2[1], A + .5*dt*k2[2], D + .5*dt*k2[3], uh)
        k4 = f(H + dt*k3[0], M + dt*k3[1], A + dt*k3[2], D + dt*k3[3], u1)
        H = H + dt/6 * (k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
        M = M + dt/6 * (k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
        A = A + dt/6 * (k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
        D = D + dt/6 * (k1[3] + 2*k2[3] + 2*k3[3] + k4[3])
        if recorder is not None:
            recorder(i, H, M, A, D)
    if mode == 'instant':
        A = p['a'] * H
    return H, M, A, D


# ---------------------------------------------------------------- equilibria at frozen D
from scipy.optimize import minimize_scalar
X_GRID = np.linspace(-25, 18, 86001)          # logit(H); H up to 1 - 1.5e-8 (finite differences stay resolvable)
H_GRID = 1 / (1 + np.exp(-X_GRID))
X_FOLD = X_GRID[::4]; H_FOLD = H_GRID[::4]      # coarser grid to locate folds, refined by minimize_scalar
GAMMA = 0.8


def u_branch(H, D=0.0, c=1.0, gamma=GAMMA, p=COMPARATOR):
    """Equilibrium forcing as a function of H* with D frozen."""
    th = p['th0'] * np.exp(-p['a'] * H)
    return th * (H / (1 - H)) ** (1 / p['n']) - hill(H, p['K'], p['q']) - gamma * D * phi(H, c)


def window(D, c, gamma=GAMMA):
    """(u_down, u_up): lower and upper fold of the branch, or (nan, nan) if it is monotone."""
    u = u_branch(H_FOLD, D, c, gamma); s = np.sign(np.diff(u))
    nz = np.nonzero(s)[0]                       # ignore exactly flat steps
    idx = nz[:-1][s[nz[:-1]] != s[nz[1:]]]
    if len(idx) < 2:
        return np.nan, np.nan
    vals = []
    for i in idx:
        lo, hi = X_FOLD[max(i - 2, 0)], X_FOLD[min(i + 3, X_FOLD.size - 1)]
        g = lambda z: u_branch(1 / (1 + np.exp(-z)), D, c, gamma)
        sgn = 1.0 if s[i] > 0 else -1.0          # slope + then -: local max (u_up); - then +: local min (u_down)
        res = minimize_scalar(lambda z: -sgn * g(z), bounds=(lo, hi), method='bounded', options=dict(xatol=1e-12))
        vals.append(float(g(res.x)))
    return min(vals), max(vals)


def critical_D(c, which, d_max=3.0):
    """Smallest D at which u_down (which='irr') or u_up (which='spont') reaches 0 at plaque u = 0."""
    k = 0 if which == 'irr' else 1
    f = lambda d: window(d, c)[k]
    grid = np.arange(0.0, d_max + 1e-9, 0.05); prev = f(grid[0])
    for d0, d1 in zip(grid[:-1], grid[1:]):
        v = f(d1)
        if not np.isfinite(v):
            return np.nan                      # folds vanished before reaching u = 0
        if v <= 0 < prev:
            return brentq(f, d0, d1, xtol=1e-12)
        prev = v
    return np.inf                               # not reached within d_max


def roots(u, D, c, gamma=GAMMA):
    f = u_branch(H_GRID, D, c, gamma) - u
    idx = np.where(np.sign(f[:-1]) != np.sign(f[1:]))[0]
    return [brentq(lambda h: u_branch(h, D, c, gamma) - u, H_GRID[i], H_GRID[i + 1], xtol=1e-14) for i in idx]


def state_at(H, D, c, gamma=GAMMA, p=COMPARATOR):
    return H, hill(H, p['K'], p['q']) + gamma * D * phi(H, c), p['a'] * H


low_equilibrium = lambda u, D, c: state_at(roots(u, D, c)[0], D, c)
high_equilibrium = lambda u, D, c: state_at(roots(u, D, c)[-1], D, c)
print("core ready")

In [ ]:
# =====================================================================
# PARAMETERS
# =====================================================================
HD = 0.8
C_LIST = [1.0, 0.5, 0.25]                 # persistent fraction of the niche (1 = Exps 09-12)
C_FINE = np.round(np.arange(0.10, 1.0001, 0.05), 3)
K_REGROW, T_MAINT = 0.05, 60.0
U_PRE = 0.20                              # plaque level of the untreated diseased site (as Exp 09)
HOSTS = [('instant', 0.0), ('adaptive', 0.03), ('adaptive', 0.01)]
R_LIST = [1.0, 0.1, 0.03]
D_GRID = np.round(np.arange(0.0, 1.0001, 0.01), 3)
THERAPIES = ['hygiene', 'debridement', 'repeated debridement']
DEBRIDE_EVERY, DEBRIDE_SESSIONS = 20.0, 10
D_RES = np.round(np.arange(0.0, 0.301, 0.05), 3)
PK = np.round(np.arange(0.20, 0.7001, 0.01), 3)
# random peaks: identical to Exp 12 (same seed, grid and order), so c = 1 reproduces it exactly
DELTA = 0.001
MEANS = np.array([0.45, 0.50, 0.55, 0.60]); CVS = np.array([0.15, 0.30])
REPS, N_CYC, SEED, ACTIVE = 100, 150, 20261015, 1e-4
if QUICK:
    C_LIST = [1.0, 0.25]; C_FINE = np.array([0.25, 0.5, 1.0]); HOSTS = HOSTS[:2]; R_LIST = [1.0]
    D_GRID = np.round(np.arange(0.0, 1.0001, 0.1), 3); D_RES = np.array([0.0, 0.2]); PK = np.round(np.arange(0.2, 0.701, 0.05), 3)
    MEANS = np.array([0.50]); CVS = np.array([0.30]); REPS, N_CYC = 10, 30

host_name = lambda mode, eps: 'instant' if mode == 'instant' else f'eps={eps}'
HCOL = {'instant': INSTANT_COL, 'eps=0.03': MEM_COL[0.03], 'eps=0.01': MEM_COL[0.01]}
CLS = {1.0: '-', 0.5: '--', 0.25: ':'}
CCOL = {1.0: INK, 0.5: "#1baf7a", 0.25: "#6250d6"}
Ns = int(round(T_MAINT / DT)); F_PK = 1 - np.exp(-K_REGROW * T_MAINT)
def periodic_u(umax):
    umax = np.asarray(umax, float)
    return lambda i, c: umax * (1 - np.exp(-K_REGROW * (((i % Ns) + c) * DT)))
print("parameters set; QUICK =", QUICK)

## A. Stage boundaries and equilibria

With $D$ frozen, equilibria satisfy $u=\theta^*(H)\,(H/(1-H))^{1/n}-G(H)-\gamma D\phi(H)$. For $c=1$ this is the lemma $u_{\text{eff}}=u+\gamma D$ and $D_{irr}=u_\downarrow/\gamma$, $D_{spont}=u_\uparrow/\gamma$. For $c<1$ the niche is smaller on the healthy branch (low $H$) than on the diseased branch, so the two boundaries move by different amounts. $D_{irr}(c)$: the diseased state first survives perfect plaque control. $D_{spont}(c)$: the healthy state is lost at perfect plaque control. Values above 1 mean the stage is never reached ($D\le1$).

In [ ]:
t0 = time.time(); rowsA = []
for c in C_FINE:
    rowsA.append(dict(c=c, D_irr=critical_D(c, 'irr'), D_spont=critical_D(c, 'spont')))
dfA = pd.DataFrame(rowsA); dfA.to_csv(EXP_DIR / "stage_boundaries.csv", index=False)
print(f"done in {time.time() - t0:.0f}s"); display(dfA.round(4))

# equilibrium check: simulated steady states at frozen D match the analytic roots
chkA = []
for c in C_LIST:
    for D in [0.1, 0.3]:
        r_ = roots(0.0, D, c)
        y0s = [state_at(r_[0], D, c), state_at(r_[-1], D, c)]
        for y0, target in zip(y0s, [r_[0], r_[-1]]):
            H, M, A, _ = run((np.array([y0[0] * 1.02]), y0[1], y0[2], D), lambda i, cc: np.zeros(1), 3000, mode='instant', gamma=GAMMA, c=c)
            chkA.append(abs(H[0] - target))
print("max |simulated - analytic| H* =", max(chkA))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
g = dfA.copy(); g['D_spont_plot'] = np.minimum(g.D_spont, 1.0)
ax[0].fill_between(g.c, 0, g.D_irr, color="#dcefe6", lw=0, label="hygiene alone suffices")
ax[0].fill_between(g.c, g.D_irr, g.D_spont_plot, color="#e7e1fb", lw=0, label="needs a perturbation of M")
ax[0].fill_between(g.c, g.D_spont_plot, 1.0, color="#f6dccf", lw=0, label="only reducing D restores health")
ax[0].plot(g.c, g.D_irr, color=INK, label=r"$D_{irr}$"); ax[0].plot(g.c, g.D_spont_plot, color=INK, ls='--', label=r"$D_{spont}$ (capped at 1)")
ax[0].set(xlabel="persistent fraction of the niche c", ylabel="attachment loss D", ylim=(0, 1.0), title="Stages at perfect plaque control")
ax[0].legend(fontsize=7)
Hs = np.linspace(1e-4, 1 - 1e-4, 4000)
for c in C_LIST:
    u = u_branch(Hs, 0.3, c); st = np.gradient(u, Hs) > 0
    ax[1].plot(np.where(st, u, np.nan), Hs, color=CCOL[c], label=f"c = {c:g}")
    ax[1].plot(np.where(~st, u, np.nan), Hs, color=CCOL[c], lw=0.7, ls='--')
ax[1].axvline(0, color=INK2, lw=0.8)
ax[1].set(xlim=(-0.4, 0.6), xlabel="plaque u", ylabel="H*", title="Equilibria at D = 0.3"); ax[1].legend(fontsize=8)
save_fig(fig, "fig13_01_stages")

## B. Treatment ladder at fixed attachment loss

As Exp 09 Section C. Diseased sites start at the high equilibrium for $u=0.20$ at each $D$ (memory at its diseased value), then receive hygiene only, one debridement ($M\to0$), or ten debridements 20 units apart, always followed by perfect plaque control. $D$ is frozen during follow-up ($\max(20/\varepsilon, 2000)$). Resolved: $H<0.5$ at the end. Reported: the largest $D$ up to which every site is resolved.

In [ ]:
def ladder(mode, eps):
    DD, RR, TH, CC = np.meshgrid(D_GRID, R_LIST, np.arange(len(THERAPIES)), C_LIST, indexing='ij')
    DD, RR, TH, CC = DD.ravel(), RR.ravel(), TH.ravel(), CC.ravel(); n = DD.size
    pre = {(d, c): high_equilibrium(U_PRE, d, c) for d in D_GRID for c in C_LIST}
    H = np.array([pre[(d, c)][0] for d, c in zip(DD, CC)]); M = np.array([pre[(d, c)][1] for d, c in zip(DD, CC)])
    A = COMPARATOR['a'] * H
    M = np.where(TH >= 1, 0.0, M)
    rep = TH == 2; every = int(round(DEBRIDE_EVERY / DT))
    def ev(i, H, M, A, Dv):
        if i > 0 and i % every == 0 and i <= every * (DEBRIDE_SESSIONS - 1):
            M = np.where(rep, 0.0, M)
        return H, M, A, Dv
    last_high = np.full(n, -1.0)
    def rec(i, H, M, A, Dv):
        last_high[H >= 0.5] = (i + 1) * DT
    follow = max(20 / eps, 2000.0) if mode == 'adaptive' else 2000.0
    Hf = run((H, M, A, DD), lambda i, c: np.zeros(n), int(round(follow / DT)), eps=eps, mode=mode,
             p=dict(COMPARATOR, r=RR), gamma=GAMMA, delta=0.0, c=CC, events=ev, recorder=rec)[0]
    return pd.DataFrame(dict(host=host_name(mode, eps), c=CC, D=DD, r=RR, therapy=[THERAPIES[t] for t in TH],
                             resolved=Hf < 0.5, H_final=Hf,
                             time_to_resolution=np.where(Hf < 0.5, np.maximum(last_high, 0.0), np.nan)))
parts = []
for mode, eps in HOSTS:
    t0 = time.time(); parts.append(ladder(mode, eps)); print(f"{host_name(mode, eps):9s} done ({time.time() - t0:.0f}s)")
dfB = pd.concat(parts, ignore_index=True); dfB.to_csv(EXP_DIR / "treatment_ladder_data.csv", index=False)

def d_max(g):
    """largest D such that every D up to it is resolved"""
    g = g.sort_values('D'); ok = g.resolved.values
    return g.D.values[np.argmin(ok) - 1] if (~ok).any() and np.argmin(ok) > 0 else (g.D.values[-1] if ok.all() else np.nan)
dmax = dfB.groupby(['host', 'c', 'r', 'therapy']).apply(d_max, include_groups=False).rename('D_max_resolved').reset_index()
dmax.to_csv(EXP_DIR / "treatment_ladder_Dmax.csv", index=False)
display(dmax.pivot_table(index=['therapy', 'r'], columns=['c', 'host'], values='D_max_resolved').round(2))

fig, ax = plt.subplots(1, len(HOSTS), figsize=(4.2 * len(HOSTS), 3.6), constrained_layout=True, sharey=True)
mk = {'hygiene': 's', 'debridement': 'o', 'repeated debridement': '^'}
for a_, (mode, eps) in zip(np.atleast_1d(ax), HOSTS):
    h = host_name(mode, eps)
    for th in THERAPIES:
        for r, ls in zip(R_LIST, ['-', '--', ':']):
            g = dmax[(dmax.host == h) & (dmax.therapy == th) & (dmax.r == r)].sort_values('c')
            a_.plot(g.c, g.D_max_resolved, mk[th] + ls, color=HCOL[h], ms=4, label=f"{th}, r = {r:g}")
    a_.plot(dfA.c, dfA.D_irr, color=INK2, lw=0.8); a_.plot(dfA.c, np.minimum(dfA.D_spont, 1), color=INK2, lw=0.8, ls='--')
    a_.set(xlabel="persistent fraction of the niche c", title=h, ylim=(0, 1.02))
np.atleast_1d(ax)[0].set_ylabel("largest D resolved"); np.atleast_1d(ax)[0].legend(fontsize=6)
save_fig(fig, "fig13_02_treatment_ladder")

## C. Maintenance after therapy

As Exp 09 Section D. Treated sites with residual attachment loss $D_{res}$ (frozen) return to periodic plaque ($T=60$), with memory fully recovered ($A_0=0$) or still at its diseased value. Output: the lowest peak plaque at which the site is re-entrapped (the tolerable peak is just below it).

In [ ]:
rowsC = []
for mode, eps in [h for h in HOSTS if h[0] == 'instant' or h[1] == 0.01] or HOSTS:
    t0 = time.time(); A_HIGH = high_equilibrium(U_PRE, 0.0, 1.0)[2]
    A0s = [0.0, A_HIGH] if mode == 'adaptive' else [0.0]
    DD, AA, PP, CC = np.meshgrid(D_RES, A0s, PK, C_LIST, indexing='ij'); DD, AA, PP, CC = DD.ravel(), AA.ravel(), PP.ravel(), CC.ravel()
    horizon = max(30 * T_MAINT, 40 / eps) if mode == 'adaptive' else 30 * T_MAINT
    nc = int(horizon / T_MAINT); Hmin = np.full((DD.size, nc), np.nan); cur = {}
    def rec(i, H, M, A, D):
        if i % Ns == 0: cur['m'] = H.copy()
        cur['m'] = np.minimum(cur['m'], H)
        if (i + 1) % Ns == 0: Hmin[:, (i + 1) // Ns - 1] = cur['m']
    run((np.zeros(DD.size), 0.0, AA, DD), periodic_u(PP / F_PK), nc * Ns, eps=eps, mode=mode, gamma=GAMMA, delta=0.0, c=CC, recorder=rec)
    conv = (Hmin[:, -3:] > 0.5).all(axis=1)
    rowsC.append(pd.DataFrame(dict(host=host_name(mode, eps), c=CC, D_res=DD, A0=AA, peak=PP, converts=conv)))
    print(f"{host_name(mode, eps):9s} done ({time.time() - t0:.0f}s)")
dfC = pd.concat(rowsC, ignore_index=True); dfC.to_csv(EXP_DIR / "post_therapy_maintenance_data.csv", index=False)
thrC = dfC[dfC.converts].groupby(['host', 'c', 'D_res', 'A0']).peak.min().rename('threshold_peak').reset_index()
thrC['memory_at_start'] = np.where(thrC.A0 > 0, 'diseased', 'recovered')
thrC.to_csv(EXP_DIR / "post_therapy_thresholds.csv", index=False)
display(thrC.pivot_table(index='D_res', columns=['c', 'host', 'memory_at_start'], values='threshold_peak').round(2))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True, sharey=True)
for a_, mem in zip(ax, ['recovered', 'diseased']):
    for (h, c), g in thrC[thrC.memory_at_start == mem].groupby(['host', 'c']):
        a_.plot(g.D_res, g.threshold_peak, 'o' + CLS[c], ms=3.5, color=HCOL[h], label=f"{h}, c = {c:g}")
    a_.set(xlabel="residual attachment loss", title=f"Memory at start of maintenance: {mem}")
ax[0].set_ylabel("re-entrapment peak plaque"); ax[0].legend(fontsize=6.5)
save_fig(fig, "fig13_03_maintenance")

## D. Random plaque peaks: spontaneous remission and episodic activity

Same random peak sequences as Exp 12 (log-normal, mean $\bar p$, CV; seed 20261015), structural memory on ($\gamma=0.8$, $\delta=0.001$, $h_D=0.8$). Two regimes: **no therapy** (plaque removed at each recall, nothing else) and **treat on detection** ($M\to0$ at a recall if $H>0.5$). *Onset*: first interval entrapped ($H>0.5$ throughout). *Remission*: a later interval that is not entrapped. *Active interval*: attachment loss $>10^{-4}$.

In [ ]:
mm, cc, rr = np.meshgrid(MEANS, CVS, np.arange(REPS), indexing='ij'); mm, cc, rr = mm.ravel(), cc.ravel(), rr.ravel()
sig = np.sqrt(np.log(1 + cc ** 2)); mu = np.log(mm) - sig ** 2 / 2
rng = np.random.default_rng(SEED)
PEAKS = np.exp(mu[:, None] + sig[:, None] * rng.standard_normal((mm.size, N_CYC)))
NS = mm.size; nC = len(C_LIST)
# tile the sites over the values of c: element j*NS + k is site k with niche C_LIST[j]
UMAX = np.tile(PEAKS / F_PK, (nC, 1)); CV_ = np.repeat(C_LIST, NS)
MM, CCV = np.tile(mm, nC), np.tile(cc, nC)

def simulate(mode, eps, treat_on_detection):
    n = UMAX.shape[0]
    Hmin = np.zeros((n, N_CYC)); D_end = np.zeros((n, N_CYC)); debrided = np.zeros((n, N_CYC), bool); cur = {}
    def u_fn(i, c):
        cyc = min(i // Ns, N_CYC - 1)
        return UMAX[:, cyc] * (1 - np.exp(-K_REGROW * (((i % Ns) + c) * DT)))
    def ev(i, H, M, A, D):
        if treat_on_detection and i > 0 and i % Ns == 0:
            fl = H > 0.5; debrided[:, i // Ns] = fl; M = np.where(fl, 0.0, M)
        return H, M, A, D
    def rec(i, H, M, A, D):
        if i % Ns == 0: cur['m'] = H.copy()
        cur['m'] = np.minimum(cur['m'], H)
        if (i + 1) % Ns == 0:
            k = (i + 1) // Ns - 1; Hmin[:, k] = cur['m']; D_end[:, k] = D
    run((np.zeros(n), 0.0, 0.0, 0.0), u_fn, N_CYC * Ns, eps=eps, mode=mode, gamma=GAMMA, delta=DELTA, hD=HD, c=CV_,
        events=ev, recorder=rec)
    dD = np.diff(np.c_[np.zeros(n), D_end], axis=1)
    return dict(Hmin=Hmin, D=D_end, dD=dD, debrided=debrided)

def runs(row):
    on, off, r, last = [], [], 0, None
    for v in row:
        if v == last: r += 1
        else:
            if last is not None: (on if last else off).append(r)
            r, last = 1, v
    if last is not None: (on if last else off).append(r)
    return on, off

def metrics(sim, sl):
    act = sim['dD'][sl] > ACTIVE; tot = sim['dD'][sl].sum(1)
    k = max(1, int(round(0.1 * sim['dD'].shape[1])))
    top = np.sort(sim['dD'][sl], 1)[:, -k:].sum(1)
    share = np.where(tot > 1e-6, top / np.maximum(tot, 1e-12), np.nan)
    ac = [np.corrcoef(r[:-1], r[1:])[0, 1] for r in act.astype(float) if 0 < r.mean() < 1 and r[:-1].std() > 0 and r[1:].std() > 0]
    on_all, off_all = [], []
    for row in act:
        on, off = runs(row); on_all += on; off_all += off
    return dict(active_fraction=act.mean(), lag1_autocorr=np.mean(ac) if ac else np.nan,
                mean_active_run=np.mean(on_all) if on_all else np.nan, mean_quiet_run=np.mean(off_all) if off_all else np.nan,
                top10_share=np.nanmean(share) if np.isfinite(share).any() else np.nan, D_final=sim['D'][sl, -1].mean())

rowsD, sims = [], {}
for treat in [False, True]:
    for mode, eps in HOSTS:
        t0 = time.time(); h = host_name(mode, eps); s = simulate(mode, eps, treat); sims[(treat, h)] = s
        ent = s['Hmin'] > 0.5
        onset = np.where(ent.any(1), ent.argmax(1), np.nan)
        remiss = np.array([(~ent[k, int(onset[k]):]).sum() if np.isfinite(onset[k]) else 0 for k in range(ent.shape[0])])
        relapses = np.array([max(len(runs(ent[k, int(onset[k]):])[0]) - 1, 0) if np.isfinite(onset[k]) else 0 for k in range(ent.shape[0])])
        for c in C_LIST:
            for m in MEANS:
                for cv in CVS:
                    sl = (CV_ == c) & (MM == m) & (CCV == cv); o = onset[sl]; conv = np.isfinite(o)
                    rowsD.append(dict(regime='treat on detection' if treat else 'no therapy', host=h, c=c, mean_peak=m, cv=cv,
                                      converted=conv.mean(), onset_median=np.nanmedian(o) if conv.any() else np.nan,
                                      sites_with_remission=(remiss[sl][conv] > 0).mean() if conv.any() else np.nan,
                                      remission_intervals_per_converted_site=remiss[sl][conv].mean() if conv.any() else np.nan,
                                      re_entrapments_per_converted_site=relapses[sl][conv].mean() if conv.any() else np.nan,
                                      debridements_per_site=s['debrided'][sl].sum(1).mean(), **metrics(s, sl)))
        print(f"{'treat' if treat else 'no therapy':10s} {h:9s} done ({time.time() - t0:.0f}s)")
dfD = pd.DataFrame(rowsD); dfD.to_csv(EXP_DIR / "random_peaks_metrics.csv", index=False)
show = ['regime', 'host', 'c', 'mean_peak', 'cv', 'converted', 'onset_median', 'sites_with_remission', 'remission_intervals_per_converted_site',
        'active_fraction', 'lag1_autocorr', 'top10_share', 'D_final']
display(dfD[show].round(3))

fig, ax = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
cv_show = CVS[-1]
for row, regime in enumerate(['no therapy', 'treat on detection']):
    for (h, c), g in dfD[(dfD.regime == regime) & (dfD.cv == cv_show)].groupby(['host', 'c']):
        kw = dict(color=HCOL[h], ls=CLS[c], marker='o', ms=3.5, label=f"{h}, c = {c:g}")
        ax[row, 0].plot(g.mean_peak, g.sites_with_remission if regime == 'no therapy' else g.active_fraction, **kw)
        ax[row, 1].plot(g.mean_peak, g.lag1_autocorr, **kw)
        ax[row, 2].plot(g.mean_peak, g.top10_share, **kw)
    ax[row, 0].set(ylabel="converted sites with a remission" if regime == 'no therapy' else "active fraction of intervals",
                   title=f"{regime}: " + ("spontaneous remission" if regime == 'no therapy' else "activity"))
    ax[row, 1].set(ylabel="lag-1 autocorrelation of activity", title=f"{regime}: clustering")
    ax[row, 2].set(ylabel="share of loss in top 10% of intervals", title=f"{regime}: burstiness")
    ax[row, 2].axhline(0.1, color=INK2, lw=0.7, ls=':')
for a_ in ax[1]: a_.set_xlabel(f"mean peak plaque (CV {cv_show:g})")
ax[0, 0].legend(fontsize=6, ncol=1)
save_fig(fig, "fig13_04_random_peaks")

# example sites without therapy: same four peak sequences, c = 1 vs the smallest c, slow memory and instant host
m_ex = 0.50 if 0.50 in MEANS else MEANS[0]
idx = np.where((mm == m_ex) & (cc == cv_show))[0][:4]
hosts_ex = [host_name(*h) for h in HOSTS if h[0] == 'instant' or h[1] == 0.01] or [host_name(*HOSTS[0])]
fig, ax = plt.subplots(len(hosts_ex), 2, figsize=(11, 2.1 * len(hosts_ex)), constrained_layout=True, sharex=True, squeeze=False)
for i, h in enumerate(hosts_ex):
    s = sims[(False, h)]
    for j, c in enumerate([C_LIST[0], C_LIST[-1]]):
        off = C_LIST.index(c) * NS
        for k, site in enumerate(idx):
            ax[i, j].plot(np.arange(N_CYC), s['D'][off + site], color=HCOL[h], lw=1.0, alpha=0.5 + 0.12 * k)
            ent = np.where(s['Hmin'][off + site] > 0.5)[0]
            ax[i, j].plot(ent, np.full(ent.size, -0.04 - 0.03 * k), '|', color=HCOL[h], ms=4)
        ax[i, j].set(ylabel="attachment loss D", title=f"No therapy, {h}, c = {c:g} (ticks: entrapped intervals)")
for a_ in ax[-1]: a_.set_xlabel("maintenance interval")
save_fig(fig, "fig13_05_example_sites_no_therapy")

## E. Checks, summary and export

In [ ]:
checks = []
b1 = dfA[dfA.c == 1.0].iloc[0] if (dfA.c == 1.0).any() else None
if b1 is not None:
    checks.append(dict(check="c = 1: D_irr = u_down/gamma and D_spont = u_up/gamma (1e-6)",
                       value=bool(abs(b1.D_irr - U_DOWN / GAMMA) < 1e-6 and abs(b1.D_spont - U_UP / GAMMA) < 1e-6)))
checks.append(dict(check="simulated equilibria at frozen D match analytic roots (1e-6)", value=bool(max(chkA) < 1e-6)))
fin = dfA[np.isfinite(dfA.D_irr)].sort_values('c')
checks.append(dict(check="D_irr non-increasing in c (a more persistent niche never protects)", value=bool(np.all(np.diff(fin.D_irr.values) <= 1e-9))))
s_all = list(sims.values())
checks.append(dict(check="D stays within [0, 1] and never decreases (random peaks)",
                   value=bool(all((s['D'] >= -1e-12).all() and (s['D'] <= 1 + 1e-9).all() and (s['dD'] >= -1e-12).all() for s in s_all))))
if not QUICK:
    # exact reproduction of earlier experiments at c = 1
    q = lambda df, **kw: df.loc[np.logical_and.reduce([df[k] == v for k, v in kw.items()])]
    hyg = q(dmax, c=1.0, therapy='hygiene').D_max_resolved
    checks.append(dict(check="c = 1 reproduces Exp 09: hygiene reach 0.07 for every host and r", value=bool(np.allclose(hyg, 0.07))))
    ref09 = {('instant', 1.0): 0.54, ('instant', 0.1): 0.58, ('eps=0.01', 1.0): 0.27, ('eps=0.01', 0.1): 0.40, ('eps=0.03', 0.1): 0.50}
    got = {k: float(q(dmax, c=1.0, therapy='debridement', host=k[0], r=k[1]).D_max_resolved.iloc[0]) for k in ref09}
    checks.append(dict(check="c = 1 reproduces Exp 09: single-debridement reach (instant r=1 0.54, r=0.1 0.58; eps=0.01 r=1 0.27, r=0.1 0.40; eps=0.03 r=0.1 0.50)",
                       value=bool(all(abs(got[k] - v) < 1e-9 for k, v in ref09.items()))))
    ref09m = {('instant', 0.0): 0.50, ('instant', 0.3): 0.25, ('eps=0.01', 0.0): 0.58, ('eps=0.01', 0.3): 0.30}
    gotm = {k: float(q(thrC, c=1.0, host=k[0], D_res=k[1], memory_at_start='recovered').threshold_peak.iloc[0]) for k in ref09m}
    checks.append(dict(check="c = 1 reproduces Exp 09: maintenance thresholds (instant 0.50 -> 0.25; eps=0.01 0.58 -> 0.30)",
                       value=bool(all(abs(gotm[k] - v) < 1e-9 for k, v in ref09m.items()))))
    a = q(dfD, regime='no therapy', host='eps=0.01', c=1.0, mean_peak=0.5, cv=0.3).iloc[0]
    b = q(dfD, regime='treat on detection', host='eps=0.01', c=1.0, mean_peak=0.5, cv=0.3).iloc[0]
    checks.append(dict(check="c = 1 reproduces Exp 12 (eps=0.01, mean 0.5, CV 0.3): converted 0.93, median onset 74, lag-1 0.5163, top-10% 0.3386",
                       value=bool(abs(a.converted - 0.93) < 1e-9 and a.onset_median == 74 and abs(a.D_final - 0.56511633) < 1e-6
                                  and abs(b.lag1_autocorr - 0.51633077) < 1e-6 and abs(b.top10_share - 0.33855683) < 1e-6)))
    checks.append(dict(check="c = 1 reproduces Exp 12: no remission without therapy in any host",
                       value=bool((q(dfD, regime='no therapy', c=1.0).remission_intervals_per_converted_site.fillna(0) == 0).all())))
dfChk = pd.DataFrame(checks); dfChk.to_csv(EXP_DIR / "checks.csv", index=False); display(dfChk)

summary = pd.concat([
    dfA.melt(id_vars='c', value_vars=['D_irr', 'D_spont']).assign(item=lambda d: d.variable + ", c=" + d.c.astype(str))[['item', 'value']],
    dmax[(dmax.therapy != 'hygiene')].assign(item=lambda d: d.therapy + " reach, " + d.host + ", r=" + d.r.astype(str) + ", c=" + d.c.astype(str)).rename(columns={'D_max_resolved': 'value'})[['item', 'value']],
    dfD[dfD.regime == 'no therapy'].assign(item=lambda d: "no therapy: converted sites with remission, " + d.host + ", c=" + d.c.astype(str) + ", mean " + d.mean_peak.astype(str) + ", CV " + d.cv.astype(str)).rename(columns={'sites_with_remission': 'value'})[['item', 'value']],
])
summary.to_csv(EXP_DIR / "summary.csv", index=False)
with open(EXP_DIR / "parameters.json", "w") as f:
    json.dump(dict(backbone=COMPARATOR, gamma=GAMMA, niche="gamma*D*(c + (1-c)*H)", c_list=C_LIST, c_fine=C_FINE.tolist(), h_D=HD,
                   u_pre_therapy=U_PRE, r_list=R_LIST, hosts=[host_name(*h) for h in HOSTS], therapies=THERAPIES,
                   debride_every=DEBRIDE_EVERY, debride_sessions=DEBRIDE_SESSIONS, d_res=D_RES.tolist(), k_regrow=K_REGROW, T=T_MAINT,
                   random_peaks=dict(delta=DELTA, means=MEANS.tolist(), cvs=CVS.tolist(), reps=REPS, n_intervals=N_CYC, seed=SEED,
                                     active_threshold=ACTIVE), dt=DT, quick=QUICK), f, indent=2)
print("saved")

In [ ]:
readme = """Experiment 13 - A pocket niche that partly resolves with inflammation
Comparator backbone. Niche term gamma*D*phi(H), phi = c + (1-c)H; c = 1 is the model of Exps 09-12, c = 0.5 and 0.25
let half or three quarters of the niche resolve with inflammation. gamma = 0.8, h_D = 0.8.
Files
- stage_boundaries.csv, fig13_01_stages                    D_irr(c), D_spont(c); equilibria at D = 0.3
- treatment_ladder_data.csv, treatment_ladder_Dmax.csv, fig13_02   hygiene / debridement / repeated debridement reach vs c
- post_therapy_maintenance_data.csv, post_therapy_thresholds.csv, fig13_03   re-entrapment peak after therapy vs D_res and c
- random_peaks_metrics.csv, fig13_04, fig13_05             Exp 12 peak sequences: spontaneous remission (no therapy), clustering,
                                                           burstiness (no therapy and treat on detection)
- checks.csv (includes exact reproduction of Exps 09 and 12 at c = 1), summary.csv, parameters.json
"""
with open(EXP_DIR / "README.txt", "w") as f:
    f.write(readme)
archive = shutil.make_archive(str(BASE_DIR / "Experiment_13_Resolving_Niche_EXPORT"), 'zip', EXP_DIR)
print("Saved:", archive)